In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

import dotenv

# 在工具当中去访问长期记忆
load_dotenv()
model=init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek"
)

In [3]:
from langgraph.prebuilt import ToolRuntime
from langchain_core.tools import tool
from typing import NotRequired
from langchain_core.messages import HumanMessage
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent, AgentState

store=InMemoryStore()
class CustomState(AgentState):
    user_id:NotRequired[str]
@tool(parse_docstring=True)
def save_user_info(name:str,runtime:ToolRuntime)->str:
    """
    将客户信息保存在长期记忆中
     
    Args:
      name:用户名
      runtime:工具的运行时
      
    Returns:
        str:保存状态
    """
    namespace=("users",)
    key=runtime.state["user_id"]
    value={"name":name}
    runtime.store.put(namespace,key,value)
    return "saved"
@tool(parse_docstring=True)
def get_user_info(runtime:ToolRuntime)->str:
    """从长期记忆中获取客户信息
    
    Args:
        runtime:工具的运行时
        
    Returns:
        str:用户信息
    """
    namespace=("users",)
    key=runtime.state["user_id"]
    item=runtime.store.get(namespace,key)
    return  str(item.value) if item else "unknown"
agent=create_agent(
    model=model,
    tools=[save_user_info,get_user_info],
    store=store,
    state_schema=CustomState,
    system_prompt="用户提及个人信息时，可以使用工具保存用户信息，如果用户询问一些信息时可以尝试使用一些工具"
)

print("="*30,'->第一个会话（线程）<-',"="*30)
response1=agent.invoke({
    "messages":[HumanMessage("你好，很高兴认识你，我叫小红")],"user_id":"user-1"
})
for msg in response1["messages"]:
    msg.pretty_print()
print("="*30,'->第二个会话（线程）<-',"="*30)
response2=agent.invoke({
    "messages":[HumanMessage("我是谁")],"user_id":"user-1"
})
for msg in response2["messages"]:
    msg.pretty_print()


============================== ->第一个会话（线程）<- ==============================
================================ Human Message =================================

你好，很高兴认识你，我叫小红
================================== Ai Message ==================================
Tool Calls:
  save_user_info (call_00_Tj2fSiBbfu0xXZEahRva1031)
 Call ID: call_00_Tj2fSiBbfu0xXZEahRva1031
  Args:
    name: 小红
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

你好，小红！很高兴认识你 😊 我已经记住了你的名字。

有什么我可以帮助你的吗？无论是生活上的问题、学习上的疑惑，还是想聊聊天，我都很乐意陪你！
============================== ->第二个会话（线程）<- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_00_q2QWOtNjDLmYiMOPoUjR5746)
 Call ID: call_00_q2QWOtNjDLmYiMOPoUjR5746
  Arg

In [ ]:
#1.2 基于postgresql数据库运行

In [4]:
from getpass import getpass
from psycopg.conninfo import make_conninfo
from langgraph.store.postgres import PostgresStore
from langgraph.prebuilt import ToolRuntime
from langchain_core.tools import tool
from typing import NotRequired
from langchain_core.messages import HumanMessage
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent, AgentState


class CustomState(AgentState):
    user_id:NotRequired[str]
@tool(parse_docstring=True)
def save_user_info(name:str,runtime:ToolRuntime)->str:
    """
    将客户信息保存在长期记忆中
     
    Args:
      name:用户名
      runtime:工具的运行时
      
    Returns:
        str:保存状态
    """
    namespace=("users",)
    key=runtime.state["user_id"]
    value={"name":name}
    runtime.store.put(namespace,key,value)
    return "saved"
@tool(parse_docstring=True)
def get_user_info(runtime:ToolRuntime)->str:
    """从长期记忆中获取客户信息
    
    Args:
        runtime:工具的运行时
        
    Returns:
        str:用户信息
    """
    namespace=("users",)
    key=runtime.state["user_id"]
    item=runtime.store.get(namespace,key)
    return  str(item.value) if item else "unknown"
DB_URL = make_conninfo(
    host="127.0.0.1",
    port=15432,
    user="langchain_user",
    password=getpass("请输入 langchain_user 的数据库密码："),
    dbname="langchain_db",
    sslmode="disable",
    gssencmode="disable",
    connect_timeout=10,
)
with PostgresStore.from_conn_string(DB_URL) as store:
    store.setup()
    
    agent=create_agent(
        model=model,
        tools=[save_user_info,get_user_info],
        store=store,
        state_schema=CustomState,
        system_prompt="用户提及个人信息时，可以使用工具保存用户信息，如果用户询问一些信息时可以尝试使用一些工具"
    )
    
    print("="*30,'->第一个会话（线程）<-',"="*30)
    response1=agent.invoke({
        "messages":[HumanMessage("你好，很高兴认识你，我叫小红")],"user_id":"user-1"
    })
    for msg in response1["messages"]:
        msg.pretty_print()
    print("="*30,'->第二个会话（线程）<-',"="*30)
    response2=agent.invoke({
        "messages":[HumanMessage("我是谁")],"user_id":"user-1"
    })
    for msg in response2["messages"]:
        msg.pretty_print()


============================== ->第一个会话（线程）<- ==============================
================================ Human Message =================================

你好，很高兴认识你，我叫小红
================================== Ai Message ==================================
Tool Calls:
  save_user_info (call_00_O79esOlzGQLd7x0DVvlQ2704)
 Call ID: call_00_O79esOlzGQLd7x0DVvlQ2704
  Args:
    name: 小红
================================= Tool Message =================================
Name: save_user_info

saved
================================== Ai Message ==================================

你好，小红！很高兴认识你 😊 我已经记住了你的名字。

请问今天有什么我可以帮你的吗？
============================== ->第二个会话（线程）<- ==============================
================================ Human Message =================================

我是谁
================================== Ai Message ==================================
Tool Calls:
  get_user_info (call_00_yUU5hIyVdb0EnpwFTKZ67006)
 Call ID: call_00_yUU5hIyVdb0EnpwFTKZ67006
  Args:
==========================